# P1 — Dataset Exploration

## 1.1 Introduction and Objectives

This project compares different NLP models in predicting ICD-defined low back pain (LBP) from emergency department (ED) chief complaints using MIMIC-IV-ED v2.2.

**Research Question**

How do different NLP models compare in their ability to predict ICD-defined low back pain from emergency department chief complaints?

Before constructing reference labels or developing NLP models, I first explore the source data to understand its structure, availability, and quality. This notebook focuses on three tables: `edstays`, `triage`, and `diagnosis`.

The objectives of this notebook are to:

- Understand the source tables and ED encounter population.
- Verify patient and encounter identifiers and cross-table relationships.
- Assess the availability and quality of chief complaints and recorded ICD diagnoses.
- Identify data quality issues that need to be addressed in subsequent data preparation.
- Prepare the raw data required for reference label construction and chief complaint analysis.

The unit of analysis is an ED encounter. This notebook examines the full ED population without preselecting LBP encounters or constructing reference labels.

The study evaluates agreement with ICD-derived reference labels rather than independently validated clinical diagnoses.

## 1.2 Data Source and Table Inventory

This study uses MIMIC-IV-ED v2.2, a de-identified dataset of emergency department encounters.

Three tables are relevant to this study:

| Table | Description | Role in This Study |
|---|---|---|
| `edstays` | ED encounter records | Defines the encounter population and patient identifiers |
| `triage` | Information recorded during ED triage | Provides chief complaint text |
| `diagnosis` | Recorded ICD diagnoses | Provides diagnosis codes for reference label construction |

I first examine the size and schema of these tables to understand their structure and identify the fields needed for subsequent data preparation.

Data source: [MIMIC-IV-ED v2.2](https://physionet.org/content/mimic-iv-ed/2.2/)

In [1]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path("../results/01_dataset_exploration")

schema = pd.read_csv(RESULTS_DIR / "table_schema.csv")
table_counts = pd.read_csv(RESULTS_DIR / "table_counts.csv")

column_counts = (
    schema.groupby("table_name")
    .size()
    .reset_index(name="column_count")
)

table_inventory = table_counts.merge(
    column_counts,
    on="table_name",
    validate="one_to_one"
)

display(table_inventory)


,table_name,row_count,column_count
0,diagnosis,899050,6
1,edstays,425087,9
2,triage,425087,11


In [2]:

relevant_fields = [
    "subject_id",
    "stay_id",
    "chiefcomplaint",
    "icd_code",
    "icd_version"
]

relevant_schema = schema.loc[
    schema["column_name"].isin(relevant_fields),
    ["table_name", "column_name", "data_type", "is_nullable"]
].reset_index(drop=True)

display(relevant_schema)

,table_name,column_name,data_type,is_nullable
0,diagnosis,subject_id,INT64,NO
1,diagnosis,stay_id,INT64,NO
2,diagnosis,icd_code,STRING,NO
3,diagnosis,icd_version,INT64,NO
4,edstays,subject_id,INT64,NO
5,edstays,stay_id,INT64,NO
6,triage,subject_id,INT64,NO
7,triage,stay_id,INT64,NO
8,triage,chiefcomplaint,STRING,YES


## 1.3 Encounter and Patient Overview

Each ED encounter represents one observation in this study. However, a patient may have multiple ED encounters. I examine the number of encounters, unique patients, and the distribution of encounters per patient to understand the study population. This also provides context for the patient-level data splitting planned for the modeling stage.

In [3]:
patient_encounters = pd.read_csv(
    RESULTS_DIR / "encounters_per_patient.csv"
)

visits = patient_encounters["encounter_count"]

total_patients = len(patient_encounters)
total_encounters = visits.sum()
repeat_patients = (visits > 1).sum()

population_summary = pd.DataFrame({
    "Metric": [
        "Total ED encounters",
        "Unique patients",
        "Mean encounters per patient",
        "Median encounters per patient",
        "Maximum encounters per patient",
        "Patients with multiple encounters",
        "Patients with multiple encounters (%)"
    ],
    "Value": [
        f"{total_encounters:,}",
        f"{total_patients:,}",
        f"{visits.mean():.2f}",
        f"{visits.median():.0f}",
        f"{visits.max():,}",
        f"{repeat_patients:,}",
        f"{repeat_patients / total_patients:.2%}"
    ]
})

display(population_summary)

,Metric,Value
0,Total ED encounters,"425,087"
1,Unique patients,"205,504"
2,Mean encounters per patient,2.07
3,Median encounters per patient,1
4,Maximum encounters per patient,321
5,Patients with multiple encounters,"69,952"
6,Patients with multiple encounters (%),34.04%


In [4]:
visit_groups = pd.cut(
    visits,
    bins=[0, 1, 2, 5, 10, float("inf")],
    labels=["1", "2", "3–5", "6–10", "11+"]
)

visit_summary = (
    visit_groups.value_counts(sort=False)
    .rename_axis("Encounters per patient")
    .reset_index(name="Patient count")
)

visit_summary["Percentage (%)"] = (
    visit_summary["Patient count"] / total_patients * 100
).round(2)

display(visit_summary)

,Encounters per patient,Patient count,Percentage (%)
0,1,135552,65.96
1,2,32672,15.90
2,3–5,25452,12.39
3,6–10,8114,3.95
4,11+,3714,1.81


In [5]:
edstays_count = table_counts.loc[
    table_counts["table_name"] == "edstays", "row_count"
].iloc[0]

assert total_encounters == edstays_count, "Encounter count mismatch"

print("Encounter count validation: PASSED")

Encounter count validation: PASSED


Approximately 34% of patients have multiple ED encounters. This supports the use of patient-grouped data splitting to prevent encounters from the same patient appearing in different data partitions.

## 1.4 Primary Keys and Table Relationships

The three source tables are linked by `stay_id`, while `subject_id` identifies patients for patient-grouped data splitting.

I perform three data quality checks before integrating the tables:

- Identifier completeness and encounter-level uniqueness.
- Composite key uniqueness (`stay_id`, `seq_num`) in the diagnosis table.
- Cross-table linkage and patient identifier consistency.

Multiple diagnosis records per encounter are expected and will be aggregated during reference label construction.

In [6]:
# Load validation results
key_validation = pd.read_csv(
    RESULTS_DIR / "key_validation.csv"
)

linkage_validation = pd.read_csv(
    RESULTS_DIR / "linkage_validation.csv"
)

# diagnosis_key_validation sql return no result, so it means no duplicate key

# Display validation results
print("Identifier and Key Validation")
display(key_validation.drop(columns="unique_patients"))

print("Diagnosis Composite Key Validation")
print(f"Duplicate key groups: 0")

print("\nCross-table Linkage Validation")
display(linkage_validation)

Identifier and Key Validation


,table_name,row_count,unique_stays,missing_stay_id,missing_subject_id
0,edstays,425087,425087,0,0
1,triage,425087,425087,0,0
2,diagnosis,899050,423989,0,0


Diagnosis Composite Key Validation
Duplicate key groups: 0

Cross-table Linkage Validation


,source_table,source_rows,unmatched_rows,subject_id_mismatches
0,triage,425087,0,0
1,diagnosis,899050,0,0


## 1.5 Chief Complaint Availability

Since chief complaint text is the only model input in this study, I check its availability across all ED encounters.

Missing values and empty or whitespace-only complaints are identified separately. Detailed text analysis and preprocessing will be performed in P3.

In [7]:
complaint_availability = pd.read_csv(
    RESULTS_DIR / "chief_complaint_availability.csv"
)

total = complaint_availability["total_encounters"].iloc[0]

availability_summary = pd.DataFrame({
    "Chief complaint status": [
        "Missing",
        "Empty or whitespace-only",
        "Available"
    ],
    "Encounter count": [
        complaint_availability["missing_count"].iloc[0],
        complaint_availability["empty_count"].iloc[0],
        complaint_availability["available_count"].iloc[0]
    ]
})

availability_summary["Percentage (%)"] = (
    availability_summary["Encounter count"] / total * 100
).round(4)

display(availability_summary)

,Chief complaint status,Encounter count,Percentage (%)
0,Missing,22,0.0052
1,Empty or whitespace-only,0,0.0000
2,Available,425065,99.9948


## 1.6 Diagnosis Availability and ICD Versions

Recorded ICD diagnoses will be used to construct the reference labels for this study.

I examine the completeness of ICD codes and their associated coding versions, as well as the distribution of ICD-9 and ICD-10 records.

Reference value set matching and label construction will be performed in P2.

In [8]:
diagnosis_availability = pd.read_csv(
    RESULTS_DIR / "diagnosis_availability.csv"
)

diagnosis_availability["diagnosis_pct"] = (
    diagnosis_availability["diagnosis_count"]
    / diagnosis_availability["diagnosis_count"].sum()
    * 100
).round(2)

diagnosis_stays = key_validation.loc[
    key_validation["table_name"] == "diagnosis",
    "unique_stays"
].iloc[0]

# Add total row
total_row = {
    "icd_version": "Total",
    "diagnosis_count": diagnosis_availability["diagnosis_count"].sum(),
    "encounter_count": diagnosis_stays,
    "missing_code_count": diagnosis_availability["missing_code_count"].sum(),
    "empty_code_count": diagnosis_availability["empty_code_count"].sum(),
    "diagnosis_pct": 100.00
}

diagnosis_summary = pd.concat(
    [diagnosis_availability, pd.DataFrame([total_row])],
    ignore_index=True
)

display(diagnosis_summary)

,icd_version,diagnosis_count,encounter_count,missing_code_count,empty_code_count,diagnosis_pct
0,9,443015,202647,0,0,49.28
1,10,456035,221342,0,0,50.72
2,Total,899050,423989,0,0,100.00


## 1.7 Cross-table Coverage

Previous sections examined chief complaint and diagnosis availability separately.

I now examine their combined coverage at the encounter level to determine how many ED encounters have both a nonempty chief complaint and recorded diagnosis information.

This provides an initial estimate of the potential modeling population before reference label construction and final eligibility checks.

In [9]:
coverage = pd.read_csv(
    RESULTS_DIR / "cross_table_coverage.csv"
)

coverage["Percentage (%)"] = (
    coverage["encounter_count"] / total_encounters * 100
).round(4)

display(coverage)

assert coverage["encounter_count"].sum() == total_encounters, \
    "Encounter count mismatch"

print("Encounter coverage validation: PASSED")

,chief_complaint_status,diagnosis_status,encounter_count,Percentage (%)
0,Available,Available,423967,99.7365
1,Available,Unavailable,1098,0.2583
2,Unavailable,Available,22,0.0052


Encounter coverage validation: PASSED


### Source Data Quality Summary

| Data Quality Check | Result |
|---|---|
| Source population | 425,087 encounters; 205,504 patients |
| Encounter key uniqueness | Verified in edstays and triage |
| Missing patient or encounter identifiers | None |
| Duplicate diagnosis composite keys | None |
| Cross-table linkage | No unmatched records or patient ID mismatches |
| Missing chief complaints | 22 encounters |
| Encounters without diagnosis records | 1,098 encounters |
| Missing or empty ICD codes | None |
| ICD versions | ICD-9 and ICD-10 |
| Combined data availability | 423,967 encounters (99.74%) |

The three source tables show consistent identifiers and reliable
cross-table linkage. Chief complaint and diagnosis information
are available for most ED encounters.

No source records have been removed or modified during this
exploration. Final eligibility will be determined in the
subsequent data preparation stages.

## 1.8 Data Extraction and Outputs

Based on the preceding data exploration and validation, I extract
the fields required for subsequent data preparation.

The three source tables are exported separately to preserve
their original granularity. No records are filtered, and no
text cleaning or reference label construction is performed
at this stage.

| Output File | Downstream Use |
|---|---|
| `edstays_raw.csv` | Encounter integration and patient-grouped splitting (P4) |
| `chief_complaint_raw.csv` | Chief complaint analysis and preprocessing (P3) |
| `diagnosis_raw.csv` | Reference label construction (P2) |

All extracted data are stored locally in `results/` and excluded
from version control.

In [10]:
# Load extracted source data
edstays_raw = pd.read_csv(
    RESULTS_DIR / "edstays_raw.csv"
)

chief_complaint_raw = pd.read_csv(
    RESULTS_DIR / "chief_complaint_raw.csv",
    dtype={"chiefcomplaint": str},
    keep_default_na=False
)

diagnosis_raw = pd.read_csv(
    RESULTS_DIR / "diagnosis_raw.csv",
    dtype={"icd_code": str},
    keep_default_na=False
)

# Validate extracted row counts against source tables
expected_counts = table_counts.set_index("table_name")["row_count"]

extraction_summary = pd.DataFrame({
    "Output file": [
        "edstays_raw.csv",
        "chief_complaint_raw.csv",
        "diagnosis_raw.csv"
    ],
    "Expected rows": [
        expected_counts["edstays"],
        expected_counts["triage"],
        expected_counts["diagnosis"]
    ],
    "Extracted rows": [
        len(edstays_raw),
        len(chief_complaint_raw),
        len(diagnosis_raw)
    ],
    "Columns": [
        edstays_raw.shape[1],
        chief_complaint_raw.shape[1],
        diagnosis_raw.shape[1]
    ]
})

extraction_summary["Match"] = (
    extraction_summary["Expected rows"]
    == extraction_summary["Extracted rows"]
)

display(extraction_summary)

assert extraction_summary["Match"].all(), "Extraction count mismatch"

print("Data extraction row count validation: PASSED")

,Output file,Expected rows,Extracted rows,Columns,Match
0,edstays_raw.csv,425087,425087,2,True
1,chief_complaint_raw.csv,425087,425087,2,True
2,diagnosis_raw.csv,899050,899050,3,True


Data extraction row count validation: PASSED
